# TP – Séance 2 : Apache Kafka – Ingestion distribuée

**Cours : Real-Time Data Engineering**  
**Enseignant : Dr. Khalil Haddaoui**

---

## Objectifs du TP

À la fin de ce TP, vous serez capables de :

- manipuler **KafkaProducer** et **KafkaConsumer** en Python ;
- configurer un producer avec différents paramètres : `acks`, `batch.size`, `linger_ms`, `compression_type` ;
- configurer un consumer avec gestion des offsets (auto-commit vs commit manuel) ;
- illustrer concrètement les sémantiques de livraison : **at-most-once**, **at-least-once** ;
- raisonner sur la notion de **lag** (retard d’un consumer) et concevoir un petit monitoring simple ;
- amorcer des **mini-projets Kafka** (fraude, clickstream, IoT) avec du code guidé.

---

## Prérequis techniques

- Python 3.x  
- Package `kafka-python` installé :  
  ```bash
  pip install kafka-python


Un cluster Kafka accessible (par exemple sur localhost:9092)

Si ce n’est pas le cas, vous pouvez malgré tout lire et compléter le code
(compréhension théorique), mais certaines cellules ne pourront pas être exécutées.

⚠️ Les cellules qui se connectent réellement à Kafka sont marquées [LIVE KAFKA].

In [10]:
pip install kafka-python

Note: you may need to restart the kernel to use updated packages.


In [11]:
#Imports et configuration de base
# Imports standard
import json
import time
import random
from datetime import datetime

# Librairie Kafka Python
try:
    from kafka import KafkaProducer, KafkaConsumer, TopicPartition
except ImportError as e:
    raise ImportError(
        "Le package 'kafka-python' n'est pas installé. "
        "Installez-le avec : pip install kafka-python"
    ) from e

# Configuration par défaut du cluster
#KAFKA_BOOTSTRAP_SERVERS = "localhost:3000"
KAFKA_BOOTSTRAP_SERVERS = "127.0.0.1:9092"


# Pour les exemples du TP
TOPIC_DEMONSTRATION = "rtde_demo_payments"
GROUP_ID_DEMONSTRATION = "rtde_demo_group"


## Partie 1 – Simulation de partitionnement par clé (sans Kafka)

Avant de nous connecter à un vrai cluster, nous allons **simuler** le partitionnement
en Python pur.

Objectif :  
- Comprendre comment la **clé** d’un message influence la **partition**.
- Lien direct avec les slides sur :
  - *partitionnement par hash(key)*,
  - *ordre garanti par clé à l’intérieur d’une partition*.


In [15]:
#fonction de partitionnement
def simple_hash_partition(key: str, num_partitions: int) -> int:
    """
    Retourne l'id de partition (entre 0 et num_partitions-1)
    en utilisant un hash simple de la clé.
    """
    return hash(key) % num_partitions


# Exemple
num_partitions = 3
keys = ["user_1", "user_2", "session_ABC", "user_1", "session_ABC"
]

for k in keys:
    p = simple_hash_partition(k, num_partitions)
    print(f"Clé={k:<12} -> partition {p}")


Clé=user_1       -> partition 2
Clé=user_2       -> partition 0
Clé=session_ABC  -> partition 0
Clé=user_1       -> partition 2
Clé=session_ABC  -> partition 0


In [13]:
#petite expérience guidée
# TODO 1 : Ajoutez plusieurs clés (user_3, user_4, user_5, etc.)
# TODO 2 : Observez ce qui se passe si vous relancez plusieurs fois cette cellule.
#          Astuce : sur une même session Python, hash() est stable, mais pas
#          forcément entre redémarrages. En pratique, Kafka utilise un hash stable
#          côté client.



more_keys = ["user_3", "user_4", "user_5", "user_1", "user_2", "user_3"]
    # TODO: ajouter des clés supplémentaires ici, par exemple :
    # "user_3", "user_4", "user_5", "user_1", "user_2", "user_3"


for k in more_keys:
    p = simple_hash_partition(k, num_partitions)
    print(f"Clé={k:<12} -> partition {p}")


Clé=user_3       -> partition 1
Clé=user_4       -> partition 2
Clé=user_5       -> partition 1
Clé=user_1       -> partition 2
Clé=user_2       -> partition 0
Clé=user_3       -> partition 1


## Partie 2 – Producer Kafka en Python

Nous allons maintenant utiliser **KafkaProducer** (librairie `kafka-python`).

Plan :

1. Producer minimal (sans options avancées).
2. Sérialisation JSON d’un événement de paiement.
3. Ajout de paramètres : `acks`, `compression_type`, `linger_ms`, `batch_size`.
4. Observations et questions (TODO) sur les compromis **fiabilité vs latence**.


In [16]:
# [LIVE KAFKA] Producer minimal, sans options avancées

producer_basic = KafkaProducer(
    bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
    # sérialisation simple des valeurs en bytes
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    key_serializer=lambda k: k.encode("utf-8") if k is not None else None,
)

print("Producer minimal créé.")


Producer minimal créé.


In [17]:
from datetime import datetime, timezone

#fonction pour créer un événement de paiement
def create_payment_event(user_id: str) -> dict:
    """
    Crée un événement de paiement factice pour nos tests.
    """
    return {
        "event_type": "payment",
        "user_id": user_id,
        "amount": round(random.uniform(10, 500), 2),
        "currency": "EUR",
        "timestamp": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
        "status": random.choice(["APPROVED", "DECLINED"]),
    }


# Exemple
create_payment_event("user_123")


{'event_type': 'payment',
 'user_id': 'user_123',
 'amount': 373.35,
 'currency': 'EUR',
 'timestamp': '2026-01-23T09:44:06.925951Z',
 'status': 'APPROVED'}

In [19]:
# [LIVE KAFKA] Envoyer 5 événements sur TOPIC_DEMONSTRATION

for i in range(5):
    user_id = f"user_{i}"
    event = create_payment_event(user_id)
    future = producer_basic.send(
        TOPIC_DEMONSTRATION,
        key=user_id,   # clé = user_id, pour conserver l'ordre par utilisateur
        value=event,
    )
    result_metadata = future.get(timeout=10)
    print(
        f"Envoyé event pour {user_id} -> topic={result_metadata.topic}, "
        f"partition={result_metadata.partition}, offset={result_metadata.offset}"
    )

producer_basic.flush()
print("Tous les messages ont été flushés.")


Envoyé event pour user_0 -> topic=rtde_demo_payments, partition=0, offset=425
Envoyé event pour user_1 -> topic=rtde_demo_payments, partition=0, offset=426
Envoyé event pour user_2 -> topic=rtde_demo_payments, partition=0, offset=427
Envoyé event pour user_3 -> topic=rtde_demo_payments, partition=0, offset=428
Envoyé event pour user_4 -> topic=rtde_demo_payments, partition=0, offset=429
Tous les messages ont été flushés.


### Producer avancé : acks, compression, batching

Nous allons maintenant configurer un producer plus **proche des cas réels** :

- `acks="all"` pour maximiser la fiabilité (attente des ISR),
- `compression_type="lz4"` (par exemple) pour réduire le volume réseau/disque,
- `linger_ms` et `batch_size` pour regrouper les messages en batchs.

**Question (à discuter)** :  
Quels effets attendus sur :
- la **latence** ?  
- le **débit** (throughput) ?  
- la **consommation CPU** ?


In [17]:
# [LIVE KAFKA] Producer avancé

producer_advanced = KafkaProducer(
    bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    key_serializer=lambda k: k.encode("utf-8") if k is not None else None,
    acks="all",               # fiabilité maximale (leader + ISR)
    compression_type="lz4",   # ou 'gzip', 'snappy', 'zstd'
    linger_ms=5,              # attendre quelques ms pour remplir les batchs
    batch_size=32_768,        # taille du batch ~32 Ko
    retries=3,                # réessais en cas d'erreur transitoire
)

print("Producer avancé créé.")


Producer avancé créé.


In [18]:
# [LIVE KAFKA] SCÉNARIO : envoi en rafale pour observer batching & performance

NUM_MESSAGES = 200

start_time = time.time()

for i in range(NUM_MESSAGES):
    user_id = f"user_{random.randint(1, 20)}"
    event = create_payment_event(user_id)
    producer_advanced.send(
        TOPIC_DEMONSTRATION,
        key=user_id,
        value=event,
    )

# Flush pour s'assurer que tout est envoyé
producer_advanced.flush()
elapsed = time.time() - start_time

print(f"{NUM_MESSAGES} messages envoyés en {elapsed:.3f} secondes")

# TODO  :
# - Que se passerait-il si linger_ms = 0 ?
# - Et si batch_size était beaucoup plus petit ?
# - Quels compromis feriez-vous pour :
#   * de la fraude bancaire (priorité fiabilité + latence faible),
#   * de la collecte de logs applicatifs (fiabilité moins critique) ?


200 messages envoyés en 0.193 secondes


Quand linger_ms est a 0, les messages sont envoyes de suite et la latence est plus faible mais avec moins de batching et la performances est plus faibles.
Si le batch_size était beaucoup petit, les messages sont envoyes en batch plus petits et le debit est reduit.
Pour le domaine de la fraude bancaire on chosit un linger_ms faible pour la latence

## Partie 3 – Consumer Kafka et gestion des offsets

Nous allons maintenant :

1. Créer un **consumer basique** (auto-commit = True).
2. Comprendre le risque d'**at-most-once** (perte possible).
3. Passer à un **commit manuel après traitement** pour illustrer **at-least-once**.
4. Discuter des **doublons** et de la nécessité d'une logique **idempotente**.


In [23]:
# [LIVE KAFKA] Consumer basique : auto_commit=True (comportement par défaut)

consumer_auto = KafkaConsumer(
    TOPIC_DEMONSTRATION,
    bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
    group_id=GROUP_ID_DEMONSTRATION + "_auto",
    auto_offset_reset="earliest",  # commencer au début si aucun offset
    enable_auto_commit=True,       # auto-commit activé
    value_deserializer=lambda v: json.loads(v.decode("utf-8")),
    key_deserializer=lambda k: k.decode("utf-8") if k is not None else None,
)

print("Consumer auto-commit créé. (ATTENTION : ceci illustre at-most-once potentiel)")


Consumer auto-commit créé. (ATTENTION : ceci illustre at-most-once potentiel)


In [24]:
# [LIVE KAFKA] Lecture de quelques messages (auto-commit)

print("Lecture de 5 messages avec auto-commit :")
count = 0
for msg in consumer_auto:
    print(
        f"[AUTO] partition={msg.partition}, offset={msg.offset}, "
        f"key={msg.key}, value={msg.value}"
    )
    count += 1
    if count >= 5:
        break


Lecture de 5 messages avec auto-commit :
[AUTO] partition=0, offset=215, key=user_0, value={'event_type': 'payment', 'user_id': 'user_0', 'amount': 174.43, 'currency': 'EUR', 'timestamp': '2026-01-23T09:08:48.739509Z', 'status': 'DECLINED'}
[AUTO] partition=0, offset=216, key=user_1, value={'event_type': 'payment', 'user_id': 'user_1', 'amount': 352.95, 'currency': 'EUR', 'timestamp': '2026-01-23T09:08:48.824942Z', 'status': 'APPROVED'}
[AUTO] partition=0, offset=217, key=user_2, value={'event_type': 'payment', 'user_id': 'user_2', 'amount': 98.11, 'currency': 'EUR', 'timestamp': '2026-01-23T09:08:48.836606Z', 'status': 'APPROVED'}
[AUTO] partition=0, offset=218, key=user_3, value={'event_type': 'payment', 'user_id': 'user_3', 'amount': 470.26, 'currency': 'EUR', 'timestamp': '2026-01-23T09:08:48.934901Z', 'status': 'DECLINED'}
[AUTO] partition=0, offset=219, key=user_4, value={'event_type': 'payment', 'user_id': 'user_4', 'amount': 278.74, 'currency': 'EUR', 'timestamp': '2026-01-23T0

In [25]:
# [LIVE KAFKA] Consumer avec commit manuel : vers at-least-once

consumer_manual = KafkaConsumer(
    TOPIC_DEMONSTRATION,
    bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
    group_id=GROUP_ID_DEMONSTRATION + "_manual",
    auto_offset_reset="earliest",
    enable_auto_commit=False,      # désactivation de l'auto-commit
    value_deserializer=lambda v: json.loads(v.decode("utf-8")),
    key_deserializer=lambda k: k.decode("utf-8") if k is not None else None,
)

print("Consumer avec commit manuel créé.")


Consumer avec commit manuel créé.


In [26]:
# [LIVE KAFKA] Exemple de boucle de consommation avec commit manuel


MAX_MESSAGES = 10
processed = 0

for msg in consumer_manual:
    try:
        print(
            f"[MANUAL] partition={msg.partition}, offset={msg.offset}, "
            f"key={msg.key}, value={msg.value}"
        )
        # Ici, on imagine un traitement métier potentiellement complexe :
        # -> détection de fraude, mise à jour de base de données, etc.

        # TODO: simuler un traitement en ajoutant un time.sleep(0.1) par ex.
        # time.sleep(0.1)
        time.sleep(0.1)


        # Si tout s'est bien passé, on commit l'offset de ce message
        consumer_manual.commit()
        processed += 1

        if processed >= MAX_MESSAGES:
            break

    except Exception as e:
        print(f"ERREUR pendant le traitement du message offset={msg.offset} : {e}")
        # Note: on ne commit PAS en cas d'erreur -> le message sera rejoué
        #       (approche at-least-once + nécessité de traitement idempotent)


[MANUAL] partition=0, offset=225, key=user_7, value={'event_type': 'payment', 'user_id': 'user_7', 'amount': 455.85, 'currency': 'EUR', 'timestamp': '2026-01-23T09:09:12.037221Z', 'status': 'DECLINED'}
[MANUAL] partition=0, offset=226, key=user_9, value={'event_type': 'payment', 'user_id': 'user_9', 'amount': 95.83, 'currency': 'EUR', 'timestamp': '2026-01-23T09:09:12.037221Z', 'status': 'DECLINED'}
[MANUAL] partition=0, offset=227, key=user_14, value={'event_type': 'payment', 'user_id': 'user_14', 'amount': 243.55, 'currency': 'EUR', 'timestamp': '2026-01-23T09:09:12.037221Z', 'status': 'APPROVED'}
[MANUAL] partition=0, offset=228, key=user_14, value={'event_type': 'payment', 'user_id': 'user_14', 'amount': 293.04, 'currency': 'EUR', 'timestamp': '2026-01-23T09:09:12.037221Z', 'status': 'DECLINED'}
[MANUAL] partition=0, offset=229, key=user_3, value={'event_type': 'payment', 'user_id': 'user_3', 'amount': 489.79, 'currency': 'EUR', 'timestamp': '2026-01-23T09:09:12.037221Z', 'status':

### at-most-once vs at-least-once

- **At-most-once** (auto-commit avant traitement, ou commit trop tôt) :
  - un crash après le commit ⇒ message **perdu** (jamais traité).
- **At-least-once** (commit après traitement) :
  - un crash après traitement mais avant le commit ⇒ message **rejoué**,
    donc duplication possible du traitement.

**Conclusion :**  
At-least-once est généralement préféré, à condition de rendre la logique métier
**idempotente** (un même message traité deux fois ne doit pas casser les données).


## Partie 4 – Lag, débit et latence : petite simulation

Dans les slides, nous avons défini le **Kafka lag** comme :

> `lag = (dernier offset produit) - (dernier offset commité)` pour une partition donnée.

Nous allons d'abord **simuler** cette notion en Python pur,
puis esquisser un monitoring avec `kafka-python`.


In [27]:
# Simulation abstraite du lag sur une partition

last_produced_offset = 0
last_committed_offset = -1  # aucun message commité au début
lags = []

for step in range(10):
    # 1) le producer envoie un nombre aléatoire de messages (entre 0 et 5)
    produced_now = random.randint(0, 5)
    last_produced_offset += produced_now

    # 2) le consumer traite et commit un certain nombre de messages (<= produits)
    consumed_now = random.randint(0, produced_now)
    last_committed_offset += consumed_now

    lag = last_produced_offset - last_committed_offset
    lags.append(lag)

    print(
        f"Étape {step:2d} | produits={produced_now:2d}, consommés={consumed_now:2d}, "
        f"offset_prod={last_produced_offset:2d}, offset_commit={last_committed_offset:2d}, "
        f"lag={lag:2d}"
    )

print("Historique du lag simulé :", lags)


Étape  0 | produits= 4, consommés= 1, offset_prod= 4, offset_commit= 0, lag= 4
Étape  1 | produits= 2, consommés= 0, offset_prod= 6, offset_commit= 0, lag= 6
Étape  2 | produits= 2, consommés= 2, offset_prod= 8, offset_commit= 2, lag= 6
Étape  3 | produits= 4, consommés= 3, offset_prod=12, offset_commit= 5, lag= 7
Étape  4 | produits= 1, consommés= 0, offset_prod=13, offset_commit= 5, lag= 8
Étape  5 | produits= 5, consommés= 3, offset_prod=18, offset_commit= 8, lag=10
Étape  6 | produits= 0, consommés= 0, offset_prod=18, offset_commit= 8, lag=10
Étape  7 | produits= 2, consommés= 2, offset_prod=20, offset_commit=10, lag=10
Étape  8 | produits= 2, consommés= 1, offset_prod=22, offset_commit=11, lag=11
Étape  9 | produits= 1, consommés= 0, offset_prod=23, offset_commit=11, lag=12
Historique du lag simulé : [4, 6, 6, 7, 8, 10, 10, 10, 11, 12]


In [28]:
#mini monitoring réel avec kafka-python
# [LIVE KAFKA] Patron de code pour récupérer la position du consumer et l'end offset
# TODO : cette cellule peut être utilisée comme base de mini-projet.

# On suppose que le consumer_manual est déjà créé plus haut.
# On part du principe qu'il est abonné à TOPIC_DEMONSTRATION.



# 1) Récupérer la liste des partitions assignées
assigned_partitions = consumer_manual.assignment()

print("Partitions assignées au consumer_manual :", assigned_partitions)

# 2) Pour chaque partition, récupérer :
#    - l'end offset (dernier offset produit) via consumer_manual.end_offsets(...)
#    - la position actuelle (offset suivant à consommer) via consumer_manual.position(...)
#    - calculer un lag approximatif = end_offset - position

for tp in assigned_partitions:
    # TODO: décommentez et complétez les lignes suivantes

    end_offsets = consumer_manual.end_offsets([tp])
    end_offset = end_offsets[tp]
    current_position = consumer_manual.position(tp)

    lag = end_offset - current_position

    print(
         f"Partition {tp.partition} | end_offset={end_offset}, "
         f"position={current_position}, lag={lag}"
     )

Partitions assignées au consumer_manual : {TopicPartition(topic='rtde_demo_payments', partition=0)}
Partition 0 | end_offset=430, position=235, lag=195


## Mini-Projet 1 – Audit simple de paiements (Kafka + Python)

**Objectif :**

- Créer un **topic** `rtde_payments_audit` (à faire en CLI ou via l’UI Kafka).
- Écrire un **producer Python** qui envoie des événements de paiement.
- Écrire un **consumer Python** avec commit manuel :
  - vérifier que **chaque paiement** reçu est bien consigné dans un log local (fichier texte par ex.),
  - raisonner sur ce qui se passe en cas de crash (où revient-on ?).

> Ce mini-projet illustre directement :  
> - topics / partitions,  
> - producers, consumers, offsets,  
> - at-least-once + idempotence simple.


In [30]:
 # [LIVE KAFKA] Producer pour le mini-projet 1

TOPIC_AUDIT = "rtde_payments_audit"

producer_audit = KafkaProducer(
    bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    key_serializer=lambda k: k.encode("utf-8") if k is not None else None,
    acks="all",
)

def send_audit_payments(num_messages: int = 20):
    """
    Envoie num_messages événements de paiement sur TOPIC_AUDIT.
    TODO: faire varier user_id, amount, status.
    """
    for i in range(num_messages):
        # TODO: générer un user_id et un event plus riche
        user_id = f"user_{i % 10}"
        event = create_payment_event(user_id)

        future = producer_audit.send(
            TOPIC_AUDIT,
            key=user_id,
            value=event,
        )
        meta = future.get(timeout=10)
        print(
            f"[AUDIT PROD] Sent to partition={meta.partition}, offset={meta.offset}, user_id={user_id}"
        )

    producer_audit.flush()
    print("Tous les messages d'audit ont été envoyés.")


# TODO: décommentez pour tester
send_audit_payments(10)


[AUDIT PROD] Sent to partition=0, offset=20, user_id=user_0
[AUDIT PROD] Sent to partition=0, offset=21, user_id=user_1
[AUDIT PROD] Sent to partition=0, offset=22, user_id=user_2
[AUDIT PROD] Sent to partition=0, offset=23, user_id=user_3
[AUDIT PROD] Sent to partition=0, offset=24, user_id=user_4
[AUDIT PROD] Sent to partition=0, offset=25, user_id=user_5
[AUDIT PROD] Sent to partition=0, offset=26, user_id=user_6
[AUDIT PROD] Sent to partition=0, offset=27, user_id=user_7
[AUDIT PROD] Sent to partition=0, offset=28, user_id=user_8
[AUDIT PROD] Sent to partition=0, offset=29, user_id=user_9
Tous les messages d'audit ont été envoyés.


In [31]:
# [LIVE KAFKA] Consumer pour le mini-projet 1 (avec commit manuel)

consumer_audit = KafkaConsumer(
    TOPIC_AUDIT,
    bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
    group_id="rtde_audit_group",
    auto_offset_reset="earliest",
    enable_auto_commit=False,
    value_deserializer=lambda v: json.loads(v.decode("utf-8")),
    key_deserializer=lambda k: k.decode("utf-8") if k is not None else None,
)

AUDIT_LOG_FILE = "audit_log_payments.txt"

def consume_and_log_audit(max_messages: int = 20):
    """
    Consomme des messages du topic d'audit, les loggue dans un fichier,
    et commit les offsets après chaque écriture.
    """
    processed = 0

    with open(AUDIT_LOG_FILE, "a", encoding="utf-8") as f:
        for msg in consumer_audit:
            event = msg.value
            line = f"{datetime.now(timezone.utc).isoformat().replace('+00:00','Z')} | partition={msg.partition} offset={msg.offset} | {event}\n"
            f.write(line)
            f.flush()

            print(f"[AUDIT CONS] {line.strip()}")

            # TODO: que se passe-t-il si on crash ici avant le commit ?
            consumer_audit.commit()
            processed += 1

            if processed >= max_messages:
                break

    print(f"{processed} messages d'audit traités et loggués.")


# TODO: décommentez pour tester
consume_and_log_audit(10)


[AUDIT CONS] 2026-01-23T10:11:30.286257Z | partition=0 offset=20 | {'event_type': 'payment', 'user_id': 'user_0', 'amount': 295.89, 'currency': 'EUR', 'timestamp': '2026-01-23T10:11:26.176107Z', 'status': 'APPROVED'}
[AUDIT CONS] 2026-01-23T10:11:30.302247Z | partition=0 offset=21 | {'event_type': 'payment', 'user_id': 'user_1', 'amount': 200.67, 'currency': 'EUR', 'timestamp': '2026-01-23T10:11:26.309341Z', 'status': 'APPROVED'}
[AUDIT CONS] 2026-01-23T10:11:30.311453Z | partition=0 offset=22 | {'event_type': 'payment', 'user_id': 'user_2', 'amount': 312.38, 'currency': 'EUR', 'timestamp': '2026-01-23T10:11:26.325293Z', 'status': 'DECLINED'}
[AUDIT CONS] 2026-01-23T10:11:30.318509Z | partition=0 offset=23 | {'event_type': 'payment', 'user_id': 'user_3', 'amount': 272.78, 'currency': 'EUR', 'timestamp': '2026-01-23T10:11:26.331108Z', 'status': 'APPROVED'}
[AUDIT CONS] 2026-01-23T10:11:30.325952Z | partition=0 offset=24 | {'event_type': 'payment', 'user_id': 'user_4', 'amount': 250.3, '

## Mini-Projet 2 – Clickstream & partitionnement

**Objectif :**

- Simuler des événements de **clickstream** (page_view, click, add_to_cart, etc.).
- Choisir une **clé métier** (par ex. `session_id` ou `user_id`) pour le partitionnement.
- Vérifier, via le producer, que les messages d'une même session arrivent **dans la même partition**.

Lien direct avec les slides :
- choix de la clé,
- ordre garanti dans une partition,
- réflexion sur "quelle vue métier je veux reconstruire ?".


In [32]:
from datetime import datetime, timezone

#générateur d’événements clickstream
PAGES = ["/home", "/search", "/product/42", "/product/99", "/cart", "/checkout"]

def create_click_event(session_id: str) -> dict:
    return {
        "event_type": "click",
        "session_id": session_id,
        "page": random.choice(PAGES),
        "timestamp": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")
,
    }

# Exemple
create_click_event("session_ABC")


{'event_type': 'click',
 'session_id': 'session_ABC',
 'page': '/cart',
 'timestamp': '2026-01-23T10:12:10.179253Z'}

In [33]:
# [LIVE KAFKA] Producer pour le mini-projet 2

TOPIC_CLICKS = "rtde_clickstream"

producer_clicks = KafkaProducer(
    bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    key_serializer=lambda k: k.encode("utf-8") if k is not None else None,
    acks=1,
)

def send_clickstream_sessions(num_sessions: int = 3, events_per_session: int = 5):
    """
    Envoie des événements de clickstream pour plusieurs sessions.
    TODO: observer la partition pour une même session_id.
    """
    for s in range(num_sessions):
        session_id = f"session_{s}"

        for _ in range(events_per_session):
            event = create_click_event(session_id)
            future = producer_clicks.send(
                TOPIC_CLICKS,
                key=session_id,     # clé = session_id (partitionnement par session)
                value=event,
            )
            meta = future.get(timeout=10)
            print(
                f"[CLICKS PROD] session_id={session_id}, "
                f"page={event['page']}, partition={meta.partition}, offset={meta.offset}"
            )

    producer_clicks.flush()
    print("Tous les événements de clickstream ont été envoyés.")


# TODO: décommentez pour tester
send_clickstream_sessions(num_sessions=3, events_per_session=5)


[CLICKS PROD] session_id=session_0, page=/home, partition=0, offset=15
[CLICKS PROD] session_id=session_0, page=/product/42, partition=0, offset=16
[CLICKS PROD] session_id=session_0, page=/product/99, partition=0, offset=17
[CLICKS PROD] session_id=session_0, page=/cart, partition=0, offset=18
[CLICKS PROD] session_id=session_0, page=/checkout, partition=0, offset=19
[CLICKS PROD] session_id=session_1, page=/product/99, partition=0, offset=20
[CLICKS PROD] session_id=session_1, page=/cart, partition=0, offset=21
[CLICKS PROD] session_id=session_1, page=/checkout, partition=0, offset=22
[CLICKS PROD] session_id=session_1, page=/product/99, partition=0, offset=23
[CLICKS PROD] session_id=session_1, page=/cart, partition=0, offset=24
[CLICKS PROD] session_id=session_2, page=/home, partition=0, offset=25
[CLICKS PROD] session_id=session_2, page=/cart, partition=0, offset=26
[CLICKS PROD] session_id=session_2, page=/product/99, partition=0, offset=27
[CLICKS PROD] session_id=session_2, pag

## Mini-Projet 3 – Mini monitoring du lag

**Objectif :**

- Se familiariser avec la notion de **lag** pour un consumer group.
- Construire un petit script qui :
  - interroge périodiquement Kafka pour récupérer `end_offset` et `position`,
  - calcule un **lag** approximatif,
  - affiche une alerte si le lag dépasse un seuil.

> Ce mini-projet prépare la séance sur le **monitoring** (Grafana / Prometheus, etc.)


In [35]:
# [LIVE KAFKA] Squelette pour un mini-monitoring de lag

MONITORED_TOPIC = TOPIC_DEMONSTRATION
MONITORED_GROUP = GROUP_ID_DEMONSTRATION + "_manual"  # par ex. le consumer_manual

# NB: kafka-python ne donne pas directement le lag d'un autre consumer group.
# Ici, on illustre le principe en utilisant un consumer "admin" positionné à la fin.

from kafka import KafkaAdminClient

def approximate_lag_for_topic(topic: str, bootstrap_servers: str = KAFKA_BOOTSTRAP_SERVERS):
    """
    Approche simplifiée :
    - crée un consumer temporaire pour lire les end_offsets d'un topic,
    - calcule le lag d'un consumer (ici, on peut le faire pour consumer_manual
      si on suppose qu'il partage les mêmes partitions).
    """

    consumer_temp = KafkaConsumer(
        bootstrap_servers=bootstrap_servers,
        enable_auto_commit=False,
    )
    consumer_temp.subscribe([topic])
    consumer_temp.poll(timeout_ms=1000)  # pour forcer l'assignation

    assigned = consumer_temp.assignment()
    print("Partitions assignées (temp) :", assigned)

    lags = {}

    for tp in assigned:
        end_offset = consumer_temp.end_offsets([tp])[tp]

        # Hypothèse : on veut comparer à la position du consumer_manual
        # TODO: gérer le cas où consumer_manual n'a pas encore de position.
        try:
            current_position = consumer_manual.position(tp)
        except Exception:
            current_position = 0

        lag = end_offset - current_position
        lags[tp] = lag

    consumer_temp.close()
    return lags


# TODO: décommentez pour tester (si consumer_manual tourne et lit le topic)
print(approximate_lag_for_topic(MONITORED_TOPIC))


Partitions assignées (temp) : {TopicPartition(topic='rtde_demo_payments', partition=0)}
{TopicPartition(topic='rtde_demo_payments', partition=0): 195}


# Conclusion du TP – Séance 2

Dans ce TP, vous avez :

- manipulé **KafkaProducer** et **KafkaConsumer** en Python ;
- configuré des paramètres essentiels côté producer :
  - `acks`, `compression_type`, `linger_ms`, `batch_size`, `retries` ;
- expérimenté la gestion des offsets côté consumer :
  - auto-commit vs commit manuel,
  - compréhension concrète de **at-most-once** et **at-least-once** ;
- simulé et commencé à monitorer le **lag** ;
- amorcé trois **mini-projets** :
  1. Audit de paiements,
  2. Clickstream & partitionnement par clé,
  3. Monitoring simple du lag.

Ces compétences s’inscrivent directement dans la logique de la **Séance 2** :
> Construire une ingestion Kafka fiable, scalable et observable.
